# 9.4 — Análise de Avaliação

Análise dos resultados de avaliação do pipeline RAG.
- Fonte: `results/eval_run_*.json` (gerado por `src/07_evaluate.py`)
- Cada run contém uma lista de dicts com campos:
  `question`, `generated_answer`, `contexts`, `final_judge_score`,
  `needs_review`, `attempts`, `judge_history`, `timing`

In [ ]:
import sys
import json
from pathlib import Path
from glob import glob

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / 'results'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('ROOT:', ROOT)
print('RESULTS_DIR:', RESULTS_DIR)

In [ ]:
# Encontra todos os arquivos eval_run_*.json e carrega o mais recente
run_files = sorted(glob(str(RESULTS_DIR / 'eval_run_*.json')))

if not run_files:
    print('Nenhum arquivo eval_run_*.json encontrado em results/.')
    print('Execute src/07_evaluate.py primeiro para gerar os resultados.')
    results = []
    df_eval = pd.DataFrame()
else:
    latest_run = run_files[-1]
    print(f'Runs disponíveis ({len(run_files)} total):')
    for f in run_files:
        print(f'  {Path(f).name}')
    print(f'\nCarregando: {Path(latest_run).name}')

    with open(latest_run, 'r', encoding='utf-8') as fh:
        results = json.load(fh)

    print(f'Perguntas no run: {len(results)}')

    # Converte para DataFrame
    df_eval = pd.DataFrame([
        {
            'question':          r.get('question', ''),
            'generated_answer':  r.get('generated_answer', r.get('answer', '')),
            'final_judge_score': r.get('final_judge_score', float('nan')),
            'needs_review':      r.get('needs_review', False),
            'attempts':          r.get('attempts', 1),
            'total_s':           r.get('timing', {}).get('total_s', float('nan')),
            'embedding_s':       r.get('timing', {}).get('embedding', float('nan')),
            'search_s':          r.get('timing', {}).get('hybrid_search', float('nan')),
            'rerank_s':          r.get('timing', {}).get('reranking', float('nan')),
            'generation_s':      r.get('timing', {}).get('generation', float('nan')),
        }
        for r in results
    ])
    print(df_eval[['final_judge_score', 'attempts', 'needs_review', 'total_s']].describe().round(3))

In [ ]:
# Distribuicao de final_judge_score
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    scores = df_eval['final_judge_score'].dropna()
    print(f'Score medio  : {scores.mean():.3f}')
    print(f'Score mediana: {scores.median():.3f}')
    print(f'Score min    : {scores.min():.3f}')
    print(f'Score max    : {scores.max():.3f}')

    fig, ax = plt.subplots()
    ax.hist(scores, bins=20, range=(0, 1), color='steelblue', edgecolor='white')
    ax.axvline(scores.mean(), color='red', linestyle='--', label=f'Media: {scores.mean():.2f}')
    ax.set_title('Distribuicao de final_judge_score')
    ax.set_xlabel('Score de faithfulness (0-1)')
    ax.set_ylabel('Frequencia')
    ax.set_xlim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Perguntas que precisam de revisao (needs_review=True)
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    nr = df_eval[df_eval['needs_review'] == True]
    total = len(df_eval)
    print(f'Perguntas com needs_review=True: {len(nr)} / {total} ({len(nr)/total*100:.1f}%)')
    print()

    if len(nr) > 0:
        print('Perguntas para revisao:')
        for i, row in nr.iterrows():
            q = str(row['question'])
            q_short = (q[:80] + '...') if len(q) > 80 else q
            print(f'  [{i}] score={row["final_judge_score"]:.2f} | {q_short}')
    else:
        print('Nenhuma pergunta marcada para revisao.')

In [ ]:
# Distribuicao de numero de tentativas
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    attempts_counts = df_eval['attempts'].value_counts().sort_index()
    print('Distribuicao de tentativas:')
    print(attempts_counts.to_string())

    fig, ax = plt.subplots(figsize=(6, 4))
    attempts_counts.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_title('Distribuicao de Numero de Tentativas por Pergunta')
    ax.set_xlabel('Tentativas')
    ax.set_ylabel('Frequencia')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top-5 perguntas com pior score
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    piores = df_eval.nsmallest(5, 'final_judge_score')[['question', 'final_judge_score', 'attempts', 'needs_review']]
    print('=== Top-5 Piores Perguntas (por score) ===')
    for i, (idx, row) in enumerate(piores.iterrows(), 1):
        q = str(row['question'])
        q_short = (q[:90] + '...') if len(q) > 90 else q
        nr_flag = ' [NEEDS REVIEW]' if row['needs_review'] else ''
        print(f'{i}. score={row["final_judge_score"]:.2f} | tentativas={row["attempts"]}{nr_flag}')
        print(f'   {q_short}')
        print()

In [ ]:
# Top-5 perguntas com melhor score
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    melhores = df_eval.nlargest(5, 'final_judge_score')[['question', 'final_judge_score', 'attempts', 'needs_review']]
    print('=== Top-5 Melhores Perguntas (por score) ===')
    for i, (idx, row) in enumerate(melhores.iterrows(), 1):
        q = str(row['question'])
        q_short = (q[:90] + '...') if len(q) > 90 else q
        print(f'{i}. score={row["final_judge_score"]:.2f} | tentativas={row["attempts"]}')
        print(f'   {q_short}')
        print()

In [ ]:
# Judge history de uma pergunta com needs_review=True
if not results:
    print('Nenhum dado disponivel.')
else:
    needs_review_items = [r for r in results if r.get('needs_review', False)]

    if not needs_review_items:
        print('Nenhuma pergunta com needs_review=True neste run.')
        print('Mostrando judge_history da pergunta com pior score como alternativa:')
        item = min(results, key=lambda r: r.get('final_judge_score', 1.0))
    else:
        # Escolhe o primeiro item com needs_review=True
        item = needs_review_items[0]

    print(f'Pergunta: {item["question"]}')
    print(f'Score final: {item.get("final_judge_score", "N/A")}')
    print(f'Tentativas: {item.get("attempts", "N/A")}')
    print(f'Needs review: {item.get("needs_review", False)}')
    print()

    judge_history = item.get('judge_history', [])
    if judge_history:
        print('=== Judge History ===')
        for j, entry in enumerate(judge_history, 1):
            print(f'  Tentativa {j}:')
            print(f'    score        : {entry.get("score", "N/A")}')
            print(f'    has_citations: {entry.get("has_citations", "N/A")}')
            print(f'    issues       : {entry.get("issues", "N/A")}')
    else:
        print('(judge_history nao disponivel neste item)')

In [ ]:
# Timing medio por etapa
if df_eval.empty:
    print('Nenhum dado disponivel.')
else:
    timing_cols = ['embedding_s', 'search_s', 'rerank_s', 'generation_s', 'total_s']
    timing_present = [c for c in timing_cols if c in df_eval.columns and df_eval[c].notna().any()]

    if not timing_present:
        print('Dados de timing nao encontrados no run.')
    else:
        timing_means = df_eval[timing_present].mean().round(3)
        print('=== Timing Medio por Etapa (segundos) ===')
        for col, val in timing_means.items():
            etapa = col.replace('_s', '')
            print(f'  {etapa:<15}: {val:.3f}s')

        display_cols = [c for c in timing_present if c != 'total_s']
        if display_cols:
            fig, ax = plt.subplots(figsize=(8, 4))
            timing_means[display_cols].sort_values(ascending=True).plot(
                kind='barh', ax=ax, color='mediumpurple', edgecolor='white'
            )
            ax.set_title('Tempo Medio por Etapa do Pipeline')
            ax.set_xlabel('Segundos')
            ax.set_yticklabels([c.replace('_s', '') for c in display_cols])
            plt.tight_layout()
            plt.show()